## MLP - Actividades de modificación

En esta notebook se evalúan las modificaciones pedidas en `Preguntas.md`:

- Dropout;
- Batch Normalization;
- Weight Decay;
- Data Augmentation;
- Early Stopping;
- inicialización Xavier;
- inicialización He/Kaiming;
- inicialización uniforme;
- logging con TensorBoard;
- visualización de histogramas de pesos.

El test fijo se mantiene excluido. Todas las comparaciones se hacen con 5-fold cross-validation sobre `trainval`.

In [ ]:
# ============================================================
# 1. Configuración de paths
# ============================================================

from pathlib import Path
import sys
import os
import json
import pandas as pd

REPO_ROOT = Path.cwd()

if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)
sys.path.append(str(REPO_ROOT / "src"))

print(f"Repo root: {REPO_ROOT}")

# ============================================================
# 2. Importar funciones de entrenamiento
# ============================================================

from mlp_training import run_cross_validation, save_experiment_outputs

In [ ]:
# ============================================================
# 3. Definición de experimentos
# ============================================================
#
# La lógica es acumulativa:
#
# MLP_1:
#   agrega Dropout.
#
# MLP_2:
#   agrega BatchNorm además de Dropout.
#
# MLP_3:
#   agrega Weight Decay además de Dropout + BatchNorm.
#
# MLP_4:
#   agrega Data Augmentation.
#
# MLP_5, MLP_6, MLP_7:
#   comparan inicializaciones manuales:
#   Xavier, He/Kaiming y uniforme.
#
# MLP_8:
#   prueba Early Stopping.
#
# Todos usan el mismo split final y la misma validación cruzada.
# ============================================================

experiments = [
    {
        "experiment": "MLP_1_dropout",
        "image_size": 64,
        "batch_size": 32,
        "epochs": 20,
        "lr": 1e-3,

        "dropout": 0.5,
        "batch_norm": False,
        "weight_decay": 0.0,
        "augmentation": "minimal",

        "initialization": "default",

        "early_stopping": False,
        "patience": 5,

        "tensorboard": True,
        "histogram_every": 5,
        "seed": 42,
    },
    {
        "experiment": "MLP_2_dropout_batchnorm",
        "image_size": 64,
        "batch_size": 32,
        "epochs": 20,
        "lr": 1e-3,

        "dropout": 0.5,
        "batch_norm": True,
        "weight_decay": 0.0,
        "augmentation": "minimal",

        "initialization": "default",

        "early_stopping": False,
        "patience": 5,

        "tensorboard": True,
        "histogram_every": 5,
        "seed": 42,
    },
    {
        "experiment": "MLP_3_dropout_batchnorm_weightdecay",
        "image_size": 64,
        "batch_size": 32,
        "epochs": 20,
        "lr": 1e-3,

        "dropout": 0.5,
        "batch_norm": True,
        "weight_decay": 1e-4,
        "augmentation": "minimal",

        "initialization": "default",

        "early_stopping": False,
        "patience": 5,

        "tensorboard": True,
        "histogram_every": 5,
        "seed": 42,
    },
    {
        "experiment": "MLP_4_data_augmentation",
        "image_size": 64,
        "batch_size": 32,
        "epochs": 20,
        "lr": 1e-3,

        "dropout": 0.5,
        "batch_norm": True,
        "weight_decay": 1e-4,
        "augmentation": "medium",

        "initialization": "default",

        "early_stopping": False,
        "patience": 5,

        "tensorboard": True,
        "histogram_every": 5,
        "seed": 42,
    },
    {
        "experiment": "MLP_5_xavier_initialization",
        "image_size": 64,
        "batch_size": 32,
        "epochs": 20,
        "lr": 1e-3,

        "dropout": 0.5,
        "batch_norm": True,
        "weight_decay": 1e-4,
        "augmentation": "medium",

        "initialization": "xavier",

        "early_stopping": False,
        "patience": 5,

        "tensorboard": True,
        "histogram_every": 5,
        "seed": 42,
    },
    {
        "experiment": "MLP_6_he_initialization",
        "image_size": 64,
        "batch_size": 32,
        "epochs": 20,
        "lr": 1e-3,

        "dropout": 0.5,
        "batch_norm": True,
        "weight_decay": 1e-4,
        "augmentation": "medium",

        "initialization": "he",

        "early_stopping": False,
        "patience": 5,

        "tensorboard": True,
        "histogram_every": 5,
        "seed": 42,
    },
    {
        "experiment": "MLP_7_uniform_initialization",
        "image_size": 64,
        "batch_size": 32,
        "epochs": 20,
        "lr": 1e-3,

        "dropout": 0.5,
        "batch_norm": True,
        "weight_decay": 1e-4,
        "augmentation": "medium",

        "initialization": "uniform",

        "early_stopping": False,
        "patience": 5,

        "tensorboard": True,
        "histogram_every": 5,
        "seed": 42,
    },
    {
        "experiment": "MLP_8_early_stopping",
        "image_size": 64,
        "batch_size": 32,
        "epochs": 30,
        "lr": 1e-3,

        "dropout": 0.5,
        "batch_norm": True,
        "weight_decay": 1e-4,
        "augmentation": "medium",

        "initialization": "he",

        "early_stopping": True,
        "patience": 5,

        "tensorboard": True,
        "histogram_every": 5,
        "seed": 42,
    },
]

In [ ]:
# ============================================================
# 4. Ejecutar experimentos
# ============================================================
#
# Cada experimento corre 5 folds.
#
# Esto puede tardar más que el baseline, pero sigue siendo MLP,
# no CNN. Si una corrida se interrumpe, se puede volver a correr
# esta celda; los archivos se sobrescriben.
# ============================================================

all_outputs = []
summary_rows = []

for config in experiments:
    print("\n" + "#" * 90)
    print(f"Ejecutando experimento: {config['experiment']}")
    print("#" * 90)

    output = run_cross_validation(
        config=config,
        split_csv="data/splits/final_split_5fold.csv",
    )

    all_outputs.append(output)
    summary_rows.append(output["summary"])

    save_experiment_outputs(
        cv_output=output,
        output_prefix=config["experiment"].lower(),
    )

In [ ]:
# ============================================================
# 5. Cargar resumen del baseline
# ============================================================
#
# El baseline se entrenó en 01_MLP_baseline.ipynb.
# Para comparar todo junto, cargamos su summary.json.
# ============================================================

baseline_summary_path = Path("experiments/mlp_0_baseline_summary.json")

if baseline_summary_path.exists():
    with open(baseline_summary_path, "r", encoding="utf-8") as f:
        baseline_summary = json.load(f)

    all_summaries = [baseline_summary] + summary_rows

else:
    print("No se encontró el baseline. Se comparan solo las modificaciones.")
    all_summaries = summary_rows

In [ ]:
# ============================================================
# 6. Tabla comparativa
# ============================================================
#
# Ordenamos por val_accuracy_mean.
# También conviene mirar macro_f1 y balanced_accuracy, porque puede
# haber desbalance de clases.
# ============================================================

summary_df = pd.DataFrame(all_summaries)

summary_df = summary_df.sort_values(
    by="val_accuracy_mean",
    ascending=False,
).reset_index(drop=True)

display(summary_df)

# ============================================================
# 7. Guardar comparación final de MLP
# ============================================================

output_csv = Path("experiments/experiments_mlp.csv")
summary_df.to_csv(output_csv, index=False)

print(f"Comparación MLP guardada en: {output_csv}")

In [ ]:
# ============================================================
# 8. Mejor configuración
# ============================================================
#
# Esta configuración NO se evalúa todavía en test.
# El test fijo se reserva para el final del TP, después de comparar
# MLP y CNN.
# ============================================================

best_mlp = summary_df.iloc[0]

print("Mejor experimento MLP según val_accuracy_mean:")
display(best_mlp)